# **Import Packages**

In [1]:
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import os
import numpy as np
from tqdm import tqdm
import glob
from textwrap import dedent

# **전처리 함수**

## **1. Make label text**

- 어떤 Object도 없는 이미지에 대한 Label text(.txt) 파일 생성 코드

In [21]:
# 정상 이미지 라벨링 text파일 생성 함수
    # 정상 이미지의 경우 라벨링 파일이 없으므로 생성
def Make_label_text(input_folder_path):
    
    print("현재 기본 경로는: ", input_folder_path, " 입니다.")
    
    # 1. 이미지 데이터의 train/val 폴더 주소
    train_images_path = os.path.join(input_folder_path, "original_images", "train")
    val_images_path = os.path.join(input_folder_path, "original_images", "val")
    
    # 2. 라벨링 데이터의 train/val 폴더 주소
    train_labels_path = os.path.join(input_folder_path, "labels", "train")
    val_labels_path = os.path.join(input_folder_path, "labels", "val")

    # 3. train 데이터
    # 3-1. train 이미지 데이터 파일명과 라벨링 데이터 파일명을 각각 불러오기 
    train_images_files = [os.path.splitext(file)[0] for file in os.listdir(train_images_path)]
    train_labels_files = [os.path.splitext(file)[0] for file in os.listdir(train_labels_path)]
    
    # 3-2.이미지 파일명 - 라벨링 파일명 = 이미지만 있는 데이터(= 정상 데이터)
    train_normal_files = list(set(train_images_files) - set(train_labels_files))
    print(f"train 이미지 중 정상 제품은 {len(train_normal_files)}개 입니다.")
    
    # 3-3. 라벨링 데이터가 없는 이미지의 경우 빈 내용의 텍스트 파일을 생성해 저장
        # 파일명: 이미지 파일명
        # 저장 위치: labels/train 폴더
    for file_name in train_normal_files:
        file_path = os.path.join(train_labels_path, f"{file_name}.txt")
        open(file_path, 'w').close()
    
    # 4. train 데이터
    # 4-1. train 이미지 데이터 파일명과 라벨링 데이터 파일명을 각각 불러오기 
    val_images_files = [os.path.splitext(file)[0] for file in os.listdir(val_images_path)]
    val_labels_files = [os.path.splitext(file)[0] for file in os.listdir(val_labels_path)]
    
    # 4-2.이미지 파일명 - 라벨링 파일명 = 이미지만 있는 데이터(= 정상 데이터)
    val_normal_files = list(set(val_images_files) - set(val_labels_files))
    print(f"val 이미지 중 정상 제품은 {len(val_normal_files)}개 입니다.")
    
    # 4-3. 라벨링 데이터가 없는 이미지의 경우 빈 내용의 텍스트 파일을 생성해 저장
        # 파일명: 이미지 파일명
        # 저장 위치: labels/val 폴더
    for file_name in val_normal_files:
        file_path = os.path.join(val_labels_path, f"{file_name}.txt")
        open(file_path, 'w').close()

    print(f"정상 이미지 라벨링 작업 끝! ================================================")

## **2. Make yaml file**

- Yolo model에 필요한 data.yaml 파일 생성 코드

In [47]:
# data.yaml 파일 생성 함수
    # 내용: 이미지(train, val)폴더 주소, 클래스 개수, 클래스 이름 리스트
def Make_yaml(input_folder_path):
    
    # 1. 경로 정보 
    output_path = os.path.join(input_folder_path, "data.yaml")        # data.yaml 파일 저장 위치
    train_path = os.path.join(input_folder_path, "images", "train")   # images/train 폴더 위치
    val_path = os.path.join(input_folder_path, "images", "val")       # images/val 폴더 위치
    
    # 2. YAML 내용 생성
    yaml_content = dedent(f"""\
    train: {train_path}  # 학습 이미지 경로
    val: {val_path}      # 검증 이미지 경로
    nc: 6                        # 클래스 개수
    names: ["A", "B", "C", "D", "E", "F"]  # 클래스 이름
    """)
    
    # 3.YAML 파일 생성
    with open(output_path, 'w', encoding = "utf-8") as file:
        file.write(yaml_content)

    print(f"data.yaml 파일 생성 끝! ================================================")

## **3. Preprocess Images**

- 이미지 전처리
- 밝기 및 대비 조절

In [ ]:
# 이미지 전처리 함수
def preprocess_images(input_path, output_path, alpha=alpha, beta=beta):
    
    # 1. 전처리할 모든 이미지 주소(input_path 내 모든 데이터)
    image_files = glob.glob(f"{input_path}/*")
    
    # 2. 모든 데이터에 접근하며 전처리 수행
    for img_path in tqdm(image_files):
        
        # 2-1. 이미지 데이터가 아닌 경우 패스
        if not img_path.lower().endswith((".jpg", ".jpeg", ".png")):
            print(f"이미지 파일이 아닙니다: {img_path}")
            continue

        # 2-2. 이미지 전처리
        img = np.array(Image.open(img_path).convert("RGB"))                   # 이미지 불러오기
        adjusted_img = cv2.convertScaleAbs(img, alpha = alpha, beta = beta)   # 이미지 밝기/대비 조절 
        adjusted_img = cv2.bilateralFilter(adjusted_img, 9, 75, 75)           # 노이즈 제거

        # 2-3. 수정된 이미지 저장
        output_img_path = os.path.join(output_path, img_path.replace(input_path, "")[1:])
        cv2.imwrite(output_img_path, adjusted_img)

# **전처리 수행**

In [ ]:
# input_folder_path = 
# output_folder_path = 

In [ ]:
# 전처리 과정을 train일때와 prediction일때를 구분

# train이면, 위 모든 작업을 수행(라벨링, yaml파일, 이미지 전처)
if train:
    Make_label_text(input_folder_path)
    Make_yaml(input_folder_path)
    
    train_folder = os.path.join(output_folder_path, "images/train")
    os.makedirs(train_folder, exist_ok = True)

    train_input_path = os.path.join(input_folder_path, "original_images/train")
    process_images(train_input_path, train_folder)

    val_folder = os.path.join(output_folder_path, "images/val")
    os.makedirs(val_folder, exist_ok = True)
    
    val_input_path = os.path.join(input_folder_path, "original_images/val")
    process_images(val_input_path, val_folder)

# prediction이면, 이미지 전처리만 수행
else:
    predict_folder = os.path.join(output_folder_path, "images")
    os.makedirs(predict_folder, exist_ok = True)

    predict_input_path = os.path.join(input_folder_path, "original_images")
    process_images(predict_input_path, predict_folder)